In [1]:
import pandas as pd
from pathlib import Path
import numpy as np


In [2]:
db_path = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales")
mercado = pd.read_csv(Path(r"D:\ProyectoAnalisisElectrico\PotencialesClientes\CalorVentasRegionales.csv"))

In [3]:
month_mapping = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "octubre": "10", "noviembre": "11", "diciembre": "12"
}

def get_folders_by_period(start_date=None, end_date=None, meses_filtro=None):
    available_folders = sorted([d.name for d in db_path.iterdir()])
    
    if start_date and end_date:
        start_str = f"{start_date[1] % 100:02d}{month_mapping[start_date[0]]}"
        end_str = f"{end_date[1] % 100:02d}{month_mapping[end_date[0]]}"
        idx_start = available_folders.index(start_str)
        idx_end = available_folders.index(end_str)
        available_folders = available_folders[idx_start : idx_end + 1]
        
    if meses_filtro:
        codigos_permitidos = [month_mapping[m.lower()] for m in meses_filtro]
        available_folders = [f for f in available_folders if f[-2:] in codigos_permitidos]
        
    return available_folders

In [4]:
estaciones_meses = {
    "Verano": ["enero", "febrero", "marzo"],
    "Otono": ["abril", "mayo", "junio"],
    "Invierno": ["julio", "agosto", "septiembre"],
    "Primavera": ["octubre", "noviembre", "diciembre"]
}
periods = get_folders_by_period(
    start_date=("mayo", 2025), 
    end_date=("abril", 2026), 
    meses_filtro=None
)

print(f"Carpetas seleccionadas: {periods}")

dfs = []
for month in periods:
    df_path = db_path / month / f"{month}_mean_month.parquet"
    dfs.append(pd.read_parquet(df_path))
    
retiros = pd.concat(dfs, ignore_index=True)


Carpetas seleccionadas: ['2505', '2506', '2507', '2508', '2509', '2510', '2511', '2512', '2601', '2602', '2603', '2604']


In [5]:
retiros.head()

,clave,nombre_barra,tension,Zona,Razon_Social,RUT,Nombre_Corto,Hora,Año_Mes,tipo,...,medida_min,CMg[CLP/KWh]_mean,CMg[CLP/KWh]_std,CMg[CLP/KWh]_count,valorizado_CLP_mean,valorizado_CLP_std,valorizado_CLP_count,medida_total,medida_mean_porcentual,medida_std_porcentual
0,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,0,2025-05-01,L_D,...,-8.96,75.958843,12.520056,31,-2329.032062,412.395645,31,-1145.877419,0.026737,-0.001281
1,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,1,2025-05-01,L_D,...,-8.82,73.731255,12.313301,31,-2276.079727,398.681717,31,-1145.877419,0.026926,-0.001211
2,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,2,2025-05-01,L_D,...,-9.10,82.363420,54.792734,31,-2607.349204,1711.523791,31,-1145.877419,0.027667,-0.001695
3,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,3,2025-05-01,L_D,...,-9.38,83.611068,61.741845,31,-2605.479383,1937.958567,31,-1145.877419,0.027210,-0.001448
4,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,4,2025-05-01,L_D,...,-8.82,76.564499,17.065693,31,-2332.254784,547.186864,31,-1145.877419,0.026536,-0.001174


In [6]:
retiros['PERIODO'] = pd.to_datetime(retiros['Año_Mes']).dt.strftime('%y%m')
print(retiros[['Año_Mes', 'PERIODO']].head())

     Año_Mes PERIODO
0 2025-05-01    2505
1 2025-05-01    2505
2 2025-05-01    2505
3 2025-05-01    2505
4 2025-05-01    2505


In [7]:
# Estandariza las variables de cruce asegurando formato de texto y eliminando espacios residuales. 
# Esto previene pérdida de datos por discrepancias tipográficas ("clave " vs "clave").
retiros['clave'] = retiros['clave'].astype(str).str.strip()
mercado['clave'] = mercado['clave'].astype(str).str.strip()

retiros['PERIODO'] = retiros['PERIODO'].astype(str).str.strip()
mercado['PERIODO'] = mercado['PERIODO'].astype(str).str.strip()

# Intersecta los registros de retiros físicos de energía con el universo de clientes identificados.
# El 'inner' join asegura mantener solo aquellos retiros cuya instalación y periodo existen en el mercado.
retiros_con_cliente = pd.merge(
    retiros,
    mercado,
    on=['clave', 'PERIODO'],
    how='inner'
)

# Renombra las columnas para reflejar la relación comercial del mercado eléctrico:
# Diferencia explícitamente entre quien inyecta/suministra la energía (Proveedor) y quien la consume (Cliente).
retiros_con_cliente = retiros_con_cliente.rename(columns={
    'RUT': 'RUT_PROVEEDOR',            # Rut de la empresa generadora/suministradora (fuente: retiros)
    'Razon_Social': 'PROVEEDOR',       # Nombre de la empresa generadora (fuente: retiros)
    'RUT_RAZON_SOCIAL': 'RUT_CLIENTE', # Identificador del consumidor industrial (fuente: mercado)
    'REGION': 'REGION_CLIENTE'         # Ubicación geográfica del punto de consumo (fuente: mercado)
})


In [8]:
# 1. Reconstrucción matemática de métricas eléctricas antes del GroupBy
# Permite recalcular promedios y varianzas de forma exacta al combinar distintos periodos temporales
metrics = ['medida', 'CMg[CLP/KWh]', 'valorizado_CLP']
for m in metrics:
    retiros_con_cliente[f'{m}_sum_val'] = retiros_con_cliente[f'{m}_mean'] * retiros_con_cliente[f'{m}_count']
    
    std_squared = retiros_con_cliente[f'{m}_std'].fillna(0) ** 2
    retiros_con_cliente[f'{m}_sum_sq'] = (retiros_con_cliente[f'{m}_count'] - 1) * std_squared + retiros_con_cliente[f'{m}_count'] * (retiros_con_cliente[f'{m}_mean'] ** 2)

# Función auxiliar para registrar el historial de valores únicos en campos de texto
def crear_log(x):
    return "::".join(x.dropna().astype(str).unique())

# Define las dimensiones para el perfil horario final: instalación, cliente, ubicación geográfica y hora
group_cols = ['clave', 'RUT_CLIENTE', 'REGION_CLIENTE', 'macrozona', 'Zona', 'Hora']

# Define las reglas de consolidación unificando las bases física, comercial y térmica
agg_rules = {
    # --- MÉTRICAS ELÉCTRICAS A RECONSTRUIR ---
    # Se suman las bases matemáticas para el cálculo posterior de estadísticos combinados
    'medida_sum_val': ('medida_sum_val', 'sum'),
    'medida_sum_sq': ('medida_sum_sq', 'sum'),
    'medida_count': ('medida_count', 'sum'),
    'medida_min': ('medida_min', 'min'), 
    
    'CMg_sum_val': ('CMg[CLP/KWh]_sum_val', 'sum'),
    'CMg_sum_sq': ('CMg[CLP/KWh]_sum_sq', 'sum'),
    'CMg_count': ('CMg[CLP/KWh]_count', 'sum'),
    
    'valorizado_sum_val': ('valorizado_CLP_sum_val', 'sum'),
    'valorizado_sum_sq': ('valorizado_CLP_sum_sq', 'sum'),
    'valorizado_count': ('valorizado_CLP_count', 'sum'),

    # --- IDENTIDAD DEL PROSPECTO TÉRMICO (Variables Estáticas) ---
    # Mantiene los datos generales del cliente. Se usa 'crear_log' para detectar si hubo 
    # cambios de sector/subsector en el tiempo.
    'CLIENTE': ('CLIENTE', 'last'),
    'CLIENTE_log': ('CLIENTE', crear_log),
    'n_clientes': ('CLIENTE', 'nunique'),
    
    'TIPO': ('TIPO', 'last'), 
    
    'NOMBRE_ESTABLECIMIENTO': ('NOMBRE_ESTABLECIMIENTO', 'last'), 
    'COMBUSTIBLE_PRIMARIO': ('COMBUSTIBLE_PRIMARIO', 'last'),
    "SECTOR": ('SECTOR', crear_log), 
    "SUBSECTOR": ('SUBSECTOR', crear_log), 
    "RUBRO": ('RUBRO', "last"), 

    # --- DEMANDA TÉRMICA (Métricas fijas) ---
    # Dado que la demanda de calor viene ya agregada, basta con extraer el último valor ('last')
    'DEMANDA_CALOR_MWH_sum': ('DEMANDA_CALOR_MWH_sum', 'last'),
    'DEMANDA_CALOR_MWH_mean': ('DEMANDA_CALOR_MWH_mean', 'last'),
    'DEMANDA_CALOR_MWH_std': ('DEMANDA_CALOR_MWH_std', 'last'),
    'DEMANDA_CALOR_MWH_max': ('DEMANDA_CALOR_MWH_max', 'last'),
    'DEMANDA_CALOR_MWH_min': ('DEMANDA_CALOR_MWH_min', 'last'),
    
    # --- TRAZABILIDAD DEL PROVEEDOR ELÉCTRICO ---
    # Registra quién suministró la energía y guarda el historial en caso de cambio de proveedor
    'RUT_PROVEEDOR': ('RUT_PROVEEDOR', 'last'),
    'RUT_PROVEEDOR_log': ('RUT_PROVEEDOR', crear_log),
    'n_rut_proveedores': ('RUT_PROVEEDOR', 'nunique'),

    'PROVEEDOR': ('PROVEEDOR', 'last'),
    'PROVEEDOR_log': ('PROVEEDOR', crear_log),
    'n_proveedores': ('PROVEEDOR', 'nunique'),
    
    # --- DATOS FÍSICOS DE LA BARRA ---
    'nombre_barra': ('nombre_barra', 'last'),
    'nombre_barra_log': ('nombre_barra', crear_log),
    'n_nombres_barra': ('nombre_barra', 'nunique'),
    
    'tension': ('tension', 'last'),
    'tension_log': ('tension', crear_log),
    'n_tensiones': ('tension', 'nunique'),
    
    'Nombre_Corto': ('Nombre_Corto', 'last'),
    'Nombre_Corto_log': ('Nombre_Corto', crear_log),
    'n_nombres_cortos': ('Nombre_Corto', 'nunique'),
    
    # --- CONTROL TEMPORAL ---
    # Mide la continuidad operativa del cliente contando cuántos meses únicos registró
    'periodo_last': ('PERIODO', 'last'),
    'periodos_log': ('PERIODO', crear_log),      
    'meses_operados': ('PERIODO', 'nunique')     
}

# 2. Ejecuta la consolidación definitiva del perfil cruzado
perfil_promedio_final = retiros_con_cliente.groupby(group_cols).agg(**agg_rules).reset_index()

# 3. Cálculo final de Promedios y Desviaciones Estándar Combinadas
prefix_map = ['medida', 'CMg', 'valorizado']

for m, prefix in zip(metrics, prefix_map):
    # Promedio Combinado global a partir de las sumas totales consolidadas
    perfil_promedio_final[f'{m}_mean'] = perfil_promedio_final[f'{prefix}_sum_val'] / perfil_promedio_final[f'{prefix}_count']
    
    # Varianza Combinada y Desviación Estándar global
    variance = (perfil_promedio_final[f'{prefix}_sum_sq'] - (perfil_promedio_final[f'{prefix}_sum_val'] ** 2 / perfil_promedio_final[f'{prefix}_count'])) / (perfil_promedio_final[f'{prefix}_count'] - 1)
    perfil_promedio_final[f'{m}_std'] = np.sqrt(np.maximum(0, variance))
    
    # Homologa el nombre de la columna de conteo SOLO si los prefijos son distintos
    if f'{m}_count' != f'{prefix}_count':
        perfil_promedio_final[f'{m}_count'] = perfil_promedio_final[f'{prefix}_count']
        perfil_promedio_final = perfil_promedio_final.drop(columns=[f'{prefix}_count'])
    
    # Limpieza: Elimina las columnas temporales usadas para el cálculo matemático
    perfil_promedio_final = perfil_promedio_final.drop(columns=[f'{prefix}_sum_val', f'{prefix}_sum_sq'])

# 4. Cálculo de consumo eléctrico total representativo
# Suma el perfil promedio de las 24 horas para obtener la energía total consumida en un día tipo
perfil_promedio_final['medida_total'] = perfil_promedio_final.groupby(['clave', 'RUT_CLIENTE', 'REGION_CLIENTE'])['medida_mean'].transform('sum')

In [9]:
perfil_promedio_final.to_parquet(Path(r"D:\ProyectoAnalisisElectrico\PotencialesClientes\PerfilesMercado.parquet"), index=False)

In [10]:
print("--- AUDITORÍA DE INTEGRIDAD EXACTA ---")

# Aísla la Hora 0 para evaluar cada instalación una sola vez y evitar multiplicar el conteo por 24 horas
auditoria = perfil_promedio_final[perfil_promedio_final['Hora'] == 0].copy()

# Función que calcula el total de días calendario sumando los días exactos de cada mes de operación ('YYMM')
def sumar_dias_calendario(periodos_log):
    periodos = str(periodos_log).split("::")
    # pd.to_datetime('%y%m') convierte el string (ej. '2601') a fecha y extrae sus días correspondientes
    return sum(pd.to_datetime(p, format='%y%m').days_in_month for p in periodos)

# Determina cuántas mediciones diarias debería tener cada cliente según los meses que estuvo activo
auditoria['dias_teoricos'] = auditoria['periodos_log'].apply(sumar_dias_calendario)

# Compara las mediciones reales almacenadas ('medida_count') frente al calendario teórico 
# para aislar cualquier instalación con datos inflados (duplicados) o lagunas (datos faltantes)
anomalias = auditoria[auditoria['medida_count'] != auditoria['dias_teoricos']]

if anomalias.empty:
    print("TEST PASADO: El conteo de mediciones cuadra perfecto con los días del calendario. No hay registros inflados ni datos faltantes.")
else:
    print(f"ALERTA: Se detectaron {len(anomalias)} perfiles con discrepancias entre mediciones y días calendario.")
    print("\nDetalle de la anomalía (revisa si medida_count es mayor o menor al teórico):")
    print(anomalias[['clave', 'RUT_CLIENTE', 'periodos_log', 'medida_count', 'dias_teoricos']].head(10))

--- AUDITORÍA DE INTEGRIDAD EXACTA ---
TEST PASADO: El conteo de mediciones cuadra perfecto con los días del calendario. No hay registros inflados ni datos faltantes.


In [11]:
perfil_promedio_final.columns

Index(['clave', 'RUT_CLIENTE', 'REGION_CLIENTE', 'macrozona', 'Zona', 'Hora',
       'medida_count', 'medida_min', 'CLIENTE', 'CLIENTE_log', 'n_clientes',
       'TIPO', 'NOMBRE_ESTABLECIMIENTO', 'COMBUSTIBLE_PRIMARIO', 'SECTOR',
       'SUBSECTOR', 'RUBRO', 'DEMANDA_CALOR_MWH_sum', 'DEMANDA_CALOR_MWH_mean',
       'DEMANDA_CALOR_MWH_std', 'DEMANDA_CALOR_MWH_max',
       'DEMANDA_CALOR_MWH_min', 'RUT_PROVEEDOR', 'RUT_PROVEEDOR_log',
       'n_rut_proveedores', 'PROVEEDOR', 'PROVEEDOR_log', 'n_proveedores',
       'nombre_barra', 'nombre_barra_log', 'n_nombres_barra', 'tension',
       'tension_log', 'n_tensiones', 'Nombre_Corto', 'Nombre_Corto_log',
       'n_nombres_cortos', 'periodo_last', 'periodos_log', 'meses_operados',
       'medida_mean', 'medida_std', 'CMg[CLP/KWh]_mean', 'CMg[CLP/KWh]_std',
       'CMg[CLP/KWh]_count', 'valorizado_CLP_mean', 'valorizado_CLP_std',
       'valorizado_CLP_count', 'medida_total'],
      dtype='str')